# Conway's Game of Life

Cellular automaton simulation on a toroidal grid, powered by the **Minkowski ECS** engine.

Each cell is an ECS entity with `CellState` and `NeighborCount` components.
Conway's rules are applied per-generation with full history recording for analysis.

## Setup

```bash
cd crates/minkowski-py
maturin develop --release
```

In [ ]:
import minkowski_py as mk
import polars as pl
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

## 1. Create and evolve the grid

In [ ]:
sim = mk.LifeSim(width=64, height=64, density=0.45)

print(f"Grid: {sim.dimensions()}")
print(f"Initial alive: {sim.alive_count()}")

sim.step(200, record=True)
print(f"After {sim.current_generation()} generations: {sim.alive_count()} alive")

## 2. Visualise the grid

In [ ]:
w, h = sim.dimensions()
grid = np.array(sim.grid_state(), dtype=np.uint8).reshape(h, w)

cmap = ListedColormap(["#1a1a2e", "#16c79a"])

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(grid, cmap=cmap, interpolation="nearest")
ax.set_title(f"Game of Life — generation {sim.current_generation()}")
ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Evolution filmstrip

Show snapshots at key generations.

In [ ]:
history = sim.history_to_polars()
n_gens = int(history["generation"].max()) + 1

# Pick 8 evenly spaced generations
gen_indices = np.linspace(0, n_gens - 1, 8, dtype=int)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
cmap = ListedColormap(["#1a1a2e", "#16c79a"])

for ax, gen_idx in zip(axes.flat, gen_indices):
    gen_data = history.filter(pl.col("generation") == gen_idx)
    grid = np.zeros((h, w), dtype=np.uint8)
    rows = gen_data["row"].to_numpy()
    cols = gen_data["col"].to_numpy()
    alive = gen_data["alive"].to_numpy()
    grid[rows, cols] = alive.astype(np.uint8)
    
    ax.imshow(grid, cmap=cmap, interpolation="nearest")
    ax.set_title(f"Gen {gen_idx}", fontsize=10)
    ax.axis("off")

plt.suptitle("Game of Life — evolution", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Population curve

In [ ]:
pop = (
    history
    .filter(pl.col("alive") == True)
    .group_by("generation")
    .agg(pl.len().alias("alive_count"))
    .sort("generation")
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(pop["generation"].to_numpy(), pop["alive_count"].to_numpy(), linewidth=1, color="#16c79a")
ax.fill_between(
    pop["generation"].to_numpy(),
    pop["alive_count"].to_numpy(),
    alpha=0.15,
    color="#16c79a",
)
ax.set_xlabel("Generation")
ax.set_ylabel("Alive cells")
ax.set_title("Population over time")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Density sweep — phase transition

How does initial density affect the steady-state population?

In [ ]:
densities = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
final_pops = []

for d in densities:
    s = mk.LifeSim(width=64, height=64, density=d)
    s.step(300)  # Let it stabilise
    final_pops.append(s.alive_count())

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(densities)), final_pops, tick_label=[f"{d:.1f}" for d in densities],
       color="#16c79a", edgecolor="#0d8a6a")
ax.set_xlabel("Initial density")
ax.set_ylabel("Alive cells after 300 generations")
ax.set_title("Steady-state population vs initial density")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 6. Activity heatmap

Which cells changed state most often? High-activity regions indicate oscillators and boundaries.

In [ ]:
# Compute per-cell toggle count across generations
history_np = history.select(["generation", "row", "col", "alive"]).to_numpy()
n_gens = int(history_np[:, 0].max()) + 1

toggle_count = np.zeros((h, w), dtype=np.int32)

# Reshape: (n_gens, h*w, 4)
cells_per_gen = h * w
for gen in range(1, n_gens):
    prev_start = (gen - 1) * cells_per_gen
    curr_start = gen * cells_per_gen
    for cell_idx in range(cells_per_gen):
        prev_alive = history_np[prev_start + cell_idx, 3]
        curr_alive = history_np[curr_start + cell_idx, 3]
        if prev_alive != curr_alive:
            r = int(history_np[curr_start + cell_idx, 1])
            c = int(history_np[curr_start + cell_idx, 2])
            toggle_count[r, c] += 1

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(toggle_count, cmap="inferno", interpolation="nearest")
plt.colorbar(im, ax=ax, label="State changes")
ax.set_title("Cell activity heatmap")
ax.axis("off")
plt.tight_layout()
plt.show()